# Compare decks with saved, frozen models

This evaluates decks using the incomplete candidate simulator. It does not train or prove a globally best Standard deck. Use matching-code episode receipts from notebook 11: each contains the deck definitions and a model checksum. Old checkpoints are rejected after simulator changes.

Set two receipt paths below. The first receipt supplies a candidate deck/model; the second supplies an opponent deck/model. You can edit the candidate/opponent dictionaries to add other explicit deck lists before running. Use evaluation seeds not used for training or selecting these decks. Each matchup runs twice per seed, switching who starts. An incomplete game prevents ranking.

The last cell starts the run. It reports start/completion, not live per-action progress. Interrupting stops the child process; incomplete comparisons are not published. Reports and the exact input plan are saved in a new run folder.


In [ ]:
from pathlib import Path
import json, subprocess, sys, uuid
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/compare_candidate_decks.py').exists()), None)
if ROOT is None: raise RuntimeError('Open from the death-knight-lab folder.')
CANDIDATE_RECEIPT = None  # Absolute path to an episode JSON from notebook 11
OPPONENT_RECEIPT = None   # Another episode receipt; may use the same file for a smoke check
SEEDS = [900001, 900002]
MAX_ACTIONS = 500
MAX_GAMES = 4
if CANDIDATE_RECEIPT is None or OPPONENT_RECEIPT is None:
    raise ValueError('Set both receipt paths before running this cell.')
left = json.loads(Path(CANDIDATE_RECEIPT).read_text())
right = json.loads(Path(OPPONENT_RECEIPT).read_text())
plan = dict(schema='deck-comparison-plan-v1',
            candidates={'candidate': left['replay_only_decks'][0]},
            opponents={'opponent': right['replay_only_decks'][1]},
            checkpoints=[left['checkpoint'], right['checkpoint']],
            seeds=SEEDS, max_actions=MAX_ACTIONS, max_games=MAX_GAMES)
required = 2 * len(SEEDS) * len(plan['candidates']) * len(plan['opponents'])
print(f'Planned games: {required}; budget: {MAX_GAMES}; no weight updates.')


In [ ]:
plan_folder = ROOT/'runs/comparison_plans'
plan_folder.mkdir(parents=True, exist_ok=True)
plan_path = plan_folder/(uuid.uuid4().hex+'.json')
plan_path.write_text(json.dumps(plan, indent=2)+'\n')
command = [sys.executable, str(ROOT/'tools/compare_candidate_decks.py'), '--engine-root', str(ROOT/'staging/rebased-88'), '--plan', str(plan_path), '--output', str(ROOT/'runs/deck_comparisons'), '--experimental']
completed = False
with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            try: item = json.loads(line)
            except ValueError:
                print(line.rstrip()); continue
            if item['event']=='started': print('Evaluation running; models remain frozen.')
            elif item['event']=='complete':
                completed = True
                print('Report:', item['report'])
                print('Ranking:', item['ranking'] if item['ranking'] is not None else 'Withheld: unfinished games')
        code = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
if code or not completed: raise RuntimeError('Comparison did not finish. No complete result is available from this run.')
